<a href="https://colab.research.google.com/github/metalBigLesson/PythonChess/blob/main/Chess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
class Game:
    def __init__(self):
        self.turn_color_sign = 1
        self.board = []
        self._place_pieces(1)
        self._place_pawns(1)
        for i in range(4):
            self.board.append([])
            for j in range(8):
                self.board[i+2].append(None)
        self._place_pawns(-1)
        self._place_pieces(-1)

    def _place_pieces(self, color_sign):
        rank = (7 - 7 * color_sign) // 2
        self.board.append([])
        self.board[rank].append(Rook(0, rank, color_sign))
        self.board[rank].append(Knight(1, rank, color_sign))
        self.board[rank].append(Bishop(2, rank, color_sign))
        self.board[rank].append(Queen(3, rank, color_sign))
        self.board[rank].append(King(4, rank, color_sign))
        self.board[rank].append(Bishop(5, rank, color_sign))
        self.board[rank].append(Knight(6, rank, color_sign))
        self.board[rank].append(Rook(7, rank, color_sign))

    def _place_pawns(self, color_sign):
        rank = (7 - 5 * color_sign) // 2
        self.board.append([])
        for i in range(8):
            self.board[rank].append(Pawn(i, rank, color_sign))

    def print_board(self):
        print('  a b c d e f g h')
        for i in range(7, -1, -1):
            print(i + 1, end=' ')
            for j in range(8):
                print(self._square_char(i, j), end='')
            print('', i + 1)
        print('  a b c d e f g h')

    def _square_char(self, x, y):
        if self.board[x][y] is None:
            if (x + y) % 2 == 0:
                return '□'
            else:
                return '■'
        else:
            return str(self.board[x][y])

    def get_piece(self, x, y):
        if (self.board[y][x]) is None:
            return Empty(x, y)
        return self.board[y][x]

    def move_piece(self, x1, y1, x2, y2):
        piece = self.get_piece(x1, y1)
        if piece.color_sign != self.turn_color_sign:
            print('잘못된 이동입니다. (상대 기물 또는 빈 셀)')
            return False
        if piece.can_move(x2, y2, self):
            piece.move(x2, y2, self)
            return True
        print('잘못된 이동입니다.')
        return False

    def input_move(self):
        while True:
            input_string = input("백: " if self.turn_color_sign else "흑: ")
            if len(input_string) != 4:
                print('잘못된 입력입니다.')
                continue
            try:
                x1 = ord(input_string[0]) - ord('a')
                y1 = int(input_string[1]) - 1
                x2 = ord(input_string[2]) - ord('a')
                y2 = int(input_string[3]) - 1
                if self.move_piece(x1, y1, x2, y2):
                    break
            except:
                print('잘못된 입력입니다.')
        self.turn_color_sign *= -1


class Piece:
    def __init__(self, x, y, color_sign):
        self.x = x
        self.y = y
        self.color_sign = color_sign #white: 1, black: -1
        self.char = ' '
        self.moved = False

    def move(self, x, y, game):
        game.board[self.y][self.x] = None
        game.board[y][x] = self
        self.x = x
        self.y = y
        self.moved = True

    def __str__(self):
        if (self.color_sign == 1):
            return self.char
        else:
            return chr(ord(self.char) + 6)

    def can_move(self, x, y, game):
        return True

class Empty(Piece):
    def __init__(self, x, y):
        super().__init__(x, y, 0)

class Pawn(Piece):
    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign)
        self.char = '♙'

    def can_move(self, x, y, game):
        x_diff = x - self.x
        y_diff_forward = (y - self.y) * self.color_sign
        if (x_diff == 0 and y_diff_forward == 1):
            return isinstance(game.get_piece(x, y), Empty) #move
        if (abs(x_diff) == 1 and y_diff_forward == 1):
            return game.get_piece(x, y).color_sign * self.color_sign == -1 #take-move
        if (x_diff == 0 and y_diff_forward == 2):
            return not self.moved and isinstance(game.get_piece(x, y), Empty)
        return False

class Knight(Piece):
    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign)
        self.char = '♘'
    def can_move(self, x, y, game):
        x_diff = x - self.x
        y_diff = y - self.y
        return x_diff * x_diff + y_diff * y_diff == 5 and self.color_sign * game.get_piece(x, y).color_sign != 1

class Bishop(Piece):
    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign)
        self.char = '♗'

    def can_move(self, x, y, game):
        x_diff = x - self.x
        y_diff = y - self.y
        if x_diff == 0 or y_diff == 0: return False
        slope = y_diff / x_diff
        if (slope == 1):
            for i in range(min(self.x, x) + 1, max(self.x, x)):
                if not isinstance(game.get_piece(i, i), Empty): return False
            return True
        if (slope == -1):
            for i in range(min(self.x, x) + 1, max(self.x, x)):
                if not isinstance(game.get_piece(i, -i), Empty): return False
            return True
        return False

class Rook(Piece):

    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign)
        self.char = '♖'

    def can_move(self, x, y, game):
        if (x == self.x and y == self.y): return False
        if (x == self.x):
            for i in range(min(self.y, y) + 1, max(self.y, y)):
                if not isinstance(game.get_piece(x, i), Empty): return False
            return True
        if (y == self.y):
            for i in range(min(self.x, x) + 1, max(self.x, x)):
                if not isinstance(game.get_piece(i, y), Empty): return False
            return True
        return False

class Queen(Piece):

    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign)
        self.char = '♕'

    def can_move(self, x, y, game):
        temp_rook = Rook(self.x, self.y, self.color_sign)
        temp_bishop = Bishop(self.x, self.y, self.color_sign)
        return temp_rook.can_move(x, y, game) or temp_bishop.can_move(x, y, game)

class King(Piece):

    def __init__(self, x, y, color_sign):
        super().__init__(x, y, color_sign)
        self.char = '♔'

    def can_move(self, x, y, game):
        if (abs(self.x - x) > 1 or abs(self.y - y) > 1): return False
        return game.get_piece(x, y).color_sign * self.color_sign != 1


game = Game()
game.print_board()
while True:
    game.input_move()
    game.print_board()


  a b c d e f g h
8 ♜♞♝♛♚♝♞♜ 8
7 ♟♟♟♟♟♟♟♟ 7
6 ■□■□■□■□ 6
5 □■□■□■□■ 5
4 ■□■□■□■□ 4
3 □■□■□■□■ 3
2 ♙♙♙♙♙♙♙♙ 2
1 ♖♘♗♕♔♗♘♖ 1
  a b c d e f g h
백: a
잘못된 입력입니다.
백: a
잘못된 입력입니다.
백: e1e1
잘못된 이동입니다.
백: e1e8
잘못된 이동입니다.
백: e9e0
잘못된 입력입니다.


KeyboardInterrupt: Interrupted by user

In [ ]:
ord('♚')

9818

입력법

기물의 원래 위치와 이동할 위치를 붙여서 입력

예) a8의 기물을 c7으로 이동: a8c7

캐슬링은 킹을 2칸 움직이는 것으로 취급

예) 킹사이드 캐슬링: e1g1


isinstance 대신 color_sign을 빈 셀 여부 확인용으로 사용 가능